# SolarVoix — Traitement incrémental d'un appel

**Notebook Fabric :** `NB_Traitement_Incremental`

**Lakehouse requis :** attachez `LH_SolarVoix` comme Lakehouse par défaut avant l'exécution.

Pour le test manuel du Bloc 1, la valeur par défaut traite `Files/audio_calls/CALL_0001.txt`.


## Cellule 0 — Installation des dépendances


In [ ]:
# === CELLULE 0 : Installation des dépendances ===

import subprocess
import sys

packages = ["transformers", "torch", "sentencepiece"]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", *packages],
    check=True,
)

print("✅ Packages installés avec succès")


## Cellule 1 — Paramètre d'entrée


In [ ]:
# === CELLULE 1 : Paramètre d'entrée ===
#
# OBJECTIF : Déclarer la variable call_file_path comme paramètre du Notebook.
#            Quand le Pipeline Data Factory appelle ce Notebook, il injecte
#            la valeur réelle du chemin de fichier via ce paramètre.
#
# IMPORTANT : Cette cellule DOIT être marquée comme "Parameters cell" dans Fabric.
# Comment faire :
#   1. Cliquer sur "..." à droite de la cellule
#   2. Sélectionner "Toggle parameter cell"
#   3. Un bandeau bleu "Parameters" apparaît en bas de la cellule
#
# call_file_path : chemin relatif du fichier, ex: "Files/audio_calls/CALL_0121.txt"
# La valeur par défaut permet de tester manuellement le Notebook sans Pipeline.

call_file_path = "Files/audio_calls/CALL_0001.txt"  # valeur par défaut pour test

print(f"Paramètre reçu : call_file_path = {call_file_path}")


## Cellule 2 — Extraction du call_id


In [ ]:
# === CELLULE 2 : Extraction du call_id ===
#
# OBJECTIF : Dériver le call_id depuis le chemin de fichier reçu en paramètre.
#
# Exemple : "Files/audio_calls/CALL_0121.txt"
#           → call_id = "CALL_0121"
#
# On utilise os.path.basename pour extraire le nom du fichier,
# puis str.replace pour supprimer l'extension.
# Robuste même si le chemin contient des sous-dossiers supplémentaires.

import os
import re

nom_fichier = os.path.basename(call_file_path)   # "CALL_0121.txt"
call_id = nom_fichier.replace(".txt", "")         # "CALL_0121"

# Validation : le call_id doit correspondre au pattern CALL_XXXX
if not re.match(r"^CALL_\d{4}$", call_id):
    raise ValueError(f"Format call_id invalide : '{call_id}'. Attendu: CALL_XXXX")

print(f"call_id extrait : {call_id}")


## Cellule 3 — Lecture et enrichissement


In [ ]:
# === CELLULE 3 : Traitement d'un seul appel (Version corrigée) ===

from transformers import pipeline as hf_pipeline
import pyspark.sql.functions as F

# ====================== VARIABLES PAR DÉFAUT ======================
# Sécurité : valeurs par défaut si la cellule est exécutée isolément
try:
    call_file_path
except NameError:
    call_file_path = "Files/audio_calls/CALL_0001.txt"

try:
    call_id
except NameError:
    # On extrait le nom du fichier comme call_id (ex: CALL_0001)
    call_id = call_file_path.split("/")[-1].replace(".txt", "")

# ====================== LECTURE DU FICHIER ======================
try:
    # mssparkutils.fs.head retourne déjà un str (UTF-8)
    transcription_text = mssparkutils.fs.head(call_file_path, 10000).strip()

    if not transcription_text:
        raise ValueError("Le fichier est vide ou ne contient que des espaces.")

    print(f"Fichier lu avec succès : {len(transcription_text)} caractères")
    print(f"Call ID : {call_id} | Chemin : {call_file_path}")

except Exception as e:
    raise FileNotFoundError(f"Impossible de lire le fichier {call_file_path} : {e}") from e

# ====================== CHARGEMENT DU MODÈLE ======================
print("Chargement du modèle de sentiment (nlptown/bert-base-multilingual-uncased-sentiment)...")
sentiment_pipeline = hf_pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    device=-1,          # CPU
    truncation=True,
    max_length=512
)

def mapper_sentiment(label: str) -> str:
    nb = int(label[0])
    if nb <= 2:
        return "negatif"
    elif nb == 3:
        return "neutre"
    else:
        return "positif"

# ====================== FONCTION PRINCIPALE ======================
def enrichir_appel(call_id: str, texte: str, duree_sec: int = 180) -> dict:
    """
    Applique l'analyse IA et calcule le score churn pour un appel unique.
    Retourne un dictionnaire prêt pour la table silver_appels_enrichis.
    """
    # Sentiment IA
    res = sentiment_pipeline([texte], batch_size=1)[0]
    sentiment_ia = mapper_sentiment(res["label"])
    score_sentiment = round(res["score"], 4)

    # Détection d'intention (mots-clés)
    KEYWORDS_INTENT = {
        "resiliation":         ["résilie", "résiliation", "annuler", "annule", "partir", "quitter"],
        "menace_juridique":    ["avocat", "plainte", "dgccrf", "médiateur", "tribunal", "juridique"],
        "churn_imminent":      ["jamais", "arnaque", "honte", "incompétent", "remboursement total"],
        "plainte_technique":   ["panne", "ne fonctionne pas", "cassé", "défaillant", "réparer"],
        "plainte_sav":         ["rappelle pas", "personne ne", "inacceptable", "scandaleux"],
        "question_facturation":["facture", "tva", "remboursement", "devis", "tarif", "prix"],
        "suivi_commande":      ["livraison", "délai", "date", "quand", "attends"],
        "satisfaction":        ["merci", "excellent", "parfait", "satisfait", "recommande"],
        "information":         ["information", "renseignement", "savoir", "comprendre"],
    }

    texte_lower = texte.lower()
    intent_detecte = "autre"
    priorite = ["resiliation", "menace_juridique", "churn_imminent",
                "plainte_technique", "plainte_sav", "question_facturation",
                "suivi_commande", "satisfaction", "information"]
    for intent in priorite:
        if any(kw in texte_lower for kw in KEYWORDS_INTENT[intent]):
            intent_detecte = intent
            break

    # Même formule que dans la Partie 1 afin qu'un même appel conserve le même score.
    score = 0
    if sentiment_ia == "negatif":
        score += 40

    mots_resil = ["résilie", "résiliation", "annuler", "quitter", "partir", "dernier"]
    if any(m in texte_lower for m in mots_resil):
        score += 30

    mots_menace = ["avocat", "plainte", "dgccrf", "médiateur", "tribunal", "assurance"]
    if any(m in texte_lower for m in mots_menace):
        score += 20

    mots_plainte = ["panne", "ne fonctionne pas", "inacceptable", "scandaleux",
                    "inadmissible", "trois fois", "cinq fois", "rappelle pas"]
    if any(m in texte_lower for m in mots_plainte):
        score += 15

    if duree_sec > 300:
        score += 10

    score_risque_churn = min(score, 100)

    return {
        "call_id":            call_id,
        "sentiment_ia":       sentiment_ia,
        "score_sentiment":    score_sentiment,
        "intent_detecte":     intent_detecte,
        "score_risque_churn": score_risque_churn,
        "alerte_critique":    score_risque_churn >= 50,
        "duree_sec":          duree_sec,           # utile de le garder
    }


# ====================== EXÉCUTION ======================
resultat = enrichir_appel(call_id, transcription_text, duree_sec=180)

print(f"\n✅ Résultat de l'enrichissement de l'appel :")
for k, v in resultat.items():
    print(f"   {k:20} : {v}")


## Cellule 4 — UPSERT Silver et alerte idempotente


In [ ]:
# === CELLULE 4 : UPSERT dans la table Silver (Delta MERGE) ===
#
# OBJECTIF : Mettre à jour la table silver_appels_enrichis pour le call_id traité.
#            Si le call_id existe déjà → UPDATE (ré-analyse).
#            Si le call_id est nouveau → INSERT.
#
# MERGE INTO est la commande Delta pour les UPSERTS.
# Plus efficace que DELETE + INSERT car Delta ne réécrit que les fichiers concernés.
#
# Note : mssparkutils.notebook.exit() retourne un JSON au Pipeline Data Factory.
# Ce JSON contient le score churn ET le chemin du fichier réellement traité.
# Le Pipeline utilise json(...).score pour la condition d'alerte
# et json(...).fichier pour identifier le fichier dans les logs Monitor.

from datetime import datetime
from pyspark.sql import Row

# Création d'un DataFrame temporaire avec le résultat
df_nouveau = spark.createDataFrame([Row(
    call_id           = resultat["call_id"],
    sentiment_ia      = resultat["sentiment_ia"],
    score_sentiment   = float(resultat["score_sentiment"]),
    intent_detecte    = resultat["intent_detecte"],
    score_risque_churn= int(resultat["score_risque_churn"]),
    alerte_critique   = bool(resultat["alerte_critique"]),
    date_mise_a_jour  = datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
)])
df_nouveau.createOrReplaceTempView("staging_appel")

# MERGE INTO : UPSERT sur call_id
spark.sql("""
    MERGE INTO silver_appels_enrichis AS target
    USING staging_appel AS source
    ON target.call_id = source.call_id
    WHEN MATCHED THEN
        UPDATE SET
            target.sentiment_ia       = source.sentiment_ia,
            target.score_sentiment    = source.score_sentiment,
            target.intent_detecte     = source.intent_detecte,
            target.score_risque_churn = source.score_risque_churn,
            target.alerte_critique    = source.alerte_critique
    WHEN NOT MATCHED THEN
        INSERT (call_id, sentiment_ia, score_sentiment,
                intent_detecte, score_risque_churn, alerte_critique)
        VALUES (source.call_id, source.sentiment_ia, source.score_sentiment,
                source.intent_detecte, source.score_risque_churn, source.alerte_critique)
""")

print(f"MERGE terminé pour {call_id}")
print(f"Score churn : {resultat['score_risque_churn']} — Alerte : {resultat['alerte_critique']}")

# -------------------------------------------------------
# ÉCRITURE DANS LA TABLE alertes_critiques (si score ≥ 50)
# -------------------------------------------------------
# Cette table est la seule "notification" nécessaire : elle est
# requêtable en SQL, visible dans Power BI, et ne dépend d'aucun
# service externe (pas de Teams, pas d'email, pas de webhook).
#
# Le Pipeline lit le score via mssparkutils.notebook.exit()
# pour afficher le statut dans le panneau Monitor de Fabric.

if resultat["alerte_critique"]:
    df_alerte = spark.createDataFrame([{
        "call_id":            resultat["call_id"],
        "score_risque_churn": int(resultat["score_risque_churn"]),
        "sentiment_ia":       resultat["sentiment_ia"],
        "intent_detecte":     resultat["intent_detecte"],
        "statut":             "NON_TRAITEE",
        "date_alerte":        datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }])
    df_alerte.createOrReplaceTempView("staging_alerte")

    if not spark.catalog.tableExists("alertes_critiques"):
        df_alerte.write.mode("overwrite").format("delta").saveAsTable("alertes_critiques")
        print(f"🚨 Table alertes_critiques créée pour {call_id}")
    else:
        spark.sql("""
            MERGE INTO alertes_critiques AS target
            USING staging_alerte AS source
            ON target.call_id = source.call_id
            WHEN MATCHED THEN
                UPDATE SET
                    target.score_risque_churn = source.score_risque_churn,
                    target.sentiment_ia       = source.sentiment_ia,
                    target.intent_detecte     = source.intent_detecte,
                    target.statut             = source.statut,
                    target.date_alerte        = source.date_alerte
            WHEN NOT MATCHED THEN
                INSERT (call_id, score_risque_churn, sentiment_ia,
                        intent_detecte, statut, date_alerte)
                VALUES (source.call_id, source.score_risque_churn, source.sentiment_ia,
                        source.intent_detecte, source.statut, source.date_alerte)
        """)
        print(f"🚨 Alerte mise à jour dans alertes_critiques pour {call_id}")
else:
    print(f"✅ Appel {call_id} traité sans alerte (score = {resultat['score_risque_churn']})")

# Retourne un JSON au Pipeline → visible dans Monitor → panneau Output
# Contient le score ET le fichier réellement traité (détecté automatiquement)
# Le Pipeline lit json(...).score pour la condition et json(...).fichier pour les logs
import json
mssparkutils.notebook.exit(json.dumps({
    "score": resultat["score_risque_churn"],
    "fichier": call_file_path
}))
